# Climate Crop Yield Intelligence 🌾🌍

I'm a data scientist and engineer, and one of my clients is an agriculture company that wants to understand how climate pressure relates to crop yield.

The question is simple:

> **Which crops and countries look more exposed to climate pressure, and can climate data improve yield forecasting?**

I use real public data for **Wheat, Maize, Rice, Potatoes, Soybeans and Barley** from **1990 to 2023**.

The project is business-first: every section starts with a practical question, then I check the data, visualize the result and explain what it means.

**IMPORTANT NOTE:** association is not causation. I use the results as signals for investigation, not proof that one variable caused another.

## Problem Statement

The client wants to understand three things:

1. How crop yield changed over time.
2. Which crops and locations look more sensitive to hotter-than-usual years.
3. Whether climate and farm-management data improve prediction beyond simple historical baselines.

The goal is not to build the most complicated model. The goal is to build a **clear, reproducible and honest analysis** that can support a business decision.

## 1. Importing Libraries

In [ ]:
from pathlib import Path
import warnings
import io
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (11, 6)
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["axes.labelsize"] = 11
pd.set_option("display.max_columns", 30)

FIG_DIR = Path("/content/climate_crop_figures") if Path("/content").exists() else Path("reports/figures")
FIG_DIR.mkdir(parents=True, exist_ok=True)
print("Libraries loaded ✅")

## 2. Importing Dataset

I load the data directly from public **Our World in Data Grapher CSV links**.

The original sources behind these series are:

- **FAO** — crop yields
- **ERA5 / Copernicus** — temperature and precipitation
- **FAO / World Bank** — fertilizer and irrigation indicators

I use **1990–2023** because this period gives a useful overlap between the crop, climate and management datasets.

No manual dataset upload is needed.

### Download note

The dataset links are public. In Google Colab, OWID can reject Python's default downloader with **HTTP 403**.

This notebook uses an explicit request header so the same public CSV links load correctly in Colab.

In [ ]:
OWID_BASE = "https://ourworldindata.org/grapher"

CROP_SLUGS = {
    "Wheat": "wheat-yields",
    "Maize": "maize-yields",
    "Rice": "rice-yields",
    "Potatoes": "potato-yields",
    "Soybeans": "soybean-yields",
    "Barley": "barley-yields",
}

DRIVER_SLUGS = {
    "temperature_c": "average-annual-surface-temperature",
    "precipitation_mm": "average-precipitation-per-year",
    "fertilizer_kg_ha": "fertilizer-use-in-kg-per-hectare-of-arable-land",
    "irrigated_land_pct": "agricultural-land-irrigation",
}

START_YEAR = 1990
END_YEAR = 2023

def owid_url(slug):
    return f"{OWID_BASE}/{slug}.csv?v=1&csvType=full&useColumnShortNames=false"

def read_owid_series(slug, value_name):
    url = owid_url(slug)

    # Colab's default urllib User-Agent can be rejected by OWID with HTTP 403.
    response = requests.get(
        url,
        headers={
            "User-Agent": "Mozilla/5.0 (compatible; ClimateCropYieldIntelligence/1.0)",
            "Accept": "text/csv,text/plain,*/*",
            "Referer": "https://ourworldindata.org/",
        },
        timeout=60,
    )
    response.raise_for_status()

    data = pd.read_csv(io.StringIO(response.text))

    id_cols = {"Entity", "Code", "Year"}
    value_cols = [c for c in data.columns if c not in id_cols]

    if len(value_cols) != 1:
        raise ValueError(
            f"Expected one value column for {slug}, found: {value_cols}"
        )

    data = data.rename(columns={value_cols[0]: value_name})
    data = data[["Entity", "Code", "Year", value_name]].copy()

    # Keep real ISO-3 country/territory rows, not OWID regional aggregates.
    data = data[
        data["Code"].astype("string").str.fullmatch(r"[A-Z]{3}", na=False)
    ].copy()

    data["Year"] = pd.to_numeric(data["Year"], errors="coerce")
    data = data.dropna(subset=["Year"])
    data["Year"] = data["Year"].astype(int)

    return data

crop_frames = []
for crop, slug in CROP_SLUGS.items():
    part = read_owid_series(slug, "yield_t_ha")
    part["crop"] = crop
    crop_frames.append(part)

yields = pd.concat(crop_frames, ignore_index=True)

driver_frames = [read_owid_series(slug, name) for name, slug in DRIVER_SLUGS.items()]
drivers = driver_frames[0]
for frame in driver_frames[1:]:
    drivers = drivers.merge(frame, on=["Entity", "Code", "Year"], how="outer")

df = yields.merge(drivers, on=["Entity", "Code", "Year"], how="left", validate="many_to_one")
df = df[df["Year"].between(START_YEAR, END_YEAR)].copy()
df = df.drop_duplicates(["Code", "Year", "crop"])
df = df.sort_values(["crop", "Code", "Year"]).reset_index(drop=True)

print("Dataset imported ✅")
print(f"Rows: {len(df):,}")

## 3. Dataset Overview

Before answering business questions, I check:

- shape and sample rows,
- country / crop / year coverage,
- missing values,
- duplicated country-year-crop rows.

In [ ]:
print("Shape:", df.shape)
print("Countries / territories:", df["Code"].nunique())
print("Crops:", df["crop"].nunique())
print("Years:", df["Year"].min(), "-", df["Year"].max())
print("Duplicated country-year-crop rows:", df.duplicated(["Code", "Year", "crop"]).sum())

display(df.head())

missing = pd.DataFrame({
    "missing_n": df.isna().sum(),
    "missing_pct": (100 * df.isna().mean()).round(2),
}).sort_values("missing_pct", ascending=False)

display(missing)

### Data quality decision

Irrigation has much lower coverage than the other variables.

I keep irrigation for **exploratory analysis only**, but I do **not** use it in the core predictive model. Mostly imputing a variable with very limited coverage could make the model look more complete than the evidence really is.

## 4. Preparing Data for Analysis

### Why these choices?

- **Country climate deviation:** compares a year with the same country's normal climate instead of comparing naturally hot and cold countries.
- **YoY yield change:** helps show short-term production movement.
- **1%–99% clipping for the YoY plot:** stops a few extreme reporting jumps from dominating the visual. The original yield values are not changed.
- **Detrending:** removes slow long-term changes in productivity and temperature before measuring their year-to-year relationship.

In [ ]:
country_climate = df[["Code", "Year", "temperature_c", "precipitation_mm"]].drop_duplicates(["Code", "Year"])
normals = country_climate.groupby("Code")[["temperature_c", "precipitation_mm"]].mean()

df["temp_deviation_c"] = df["temperature_c"] - df["Code"].map(normals["temperature_c"])
df["precip_deviation_mm"] = df["precipitation_mm"] - df["Code"].map(normals["precipitation_mm"])

ordered = df.sort_values(["Code", "crop", "Year"]).copy()
df["yield_yoy_pct"] = ordered.groupby(["Code", "crop"])["yield_t_ha"].pct_change(fill_method=None).mul(100).reindex(df.index)

# 1%–99% clipping is only for readable YoY visualizations.
lo, hi = df["yield_yoy_pct"].quantile([0.01, 0.99])
df["yield_yoy_pct_w"] = df["yield_yoy_pct"].clip(lo, hi)

print("Analysis features ready ✅")

In [ ]:
def linear_residual(year, values):
    coef = np.polyfit(year.astype(float), values.astype(float), deg=1)
    return values.astype(float) - np.polyval(coef, year.astype(float))

def add_detrended_residuals(data, min_observations=8):
    out = data.copy()
    out["temp_detrended_c"] = np.nan
    out["yield_detrended_t_ha"] = np.nan

    # 8 observations prevents fitting a trend to very short histories.
    for (_, _), part in out.groupby(["Code", "crop"]):
        valid = part.dropna(subset=["Year", "temperature_c", "yield_t_ha"])
        if len(valid) < min_observations:
            continue
        years = valid["Year"].to_numpy(float)
        out.loc[valid.index, "temp_detrended_c"] = linear_residual(years, valid["temperature_c"].to_numpy(float))
        out.loc[valid.index, "yield_detrended_t_ha"] = linear_residual(years, valid["yield_t_ha"].to_numpy(float))
    return out

detrended = add_detrended_residuals(df)

## 5. Business Questions

### 1. Which crops improved the most since 1990?

In [ ]:
trend = df.groupby(["crop", "Year"], as_index=False)["yield_t_ha"].median()
plt.figure(figsize=(12, 6))
sns.lineplot(data=trend, x="Year", y="yield_t_ha", hue="crop", palette="Set2", linewidth=2)
plt.title("Median Crop Yield Across Countries (1990–2023)")
plt.ylabel("Yield (t/ha)")
plt.xlabel("Year")
plt.tight_layout()
plt.savefig(FIG_DIR / "01_yield_trends.png", dpi=180, bbox_inches="tight")
plt.show()

early = df[df["Year"].between(1990, 1994)].groupby("crop")["yield_t_ha"].median()
recent = df[df["Year"].between(2019, 2023)].groupby("crop")["yield_t_ha"].median()
yield_change = pd.concat([early.rename("1990_1994"), recent.rename("2019_2023")], axis=1).dropna()
yield_change["change_pct"] = 100 * (yield_change["2019_2023"] - yield_change["1990_1994"]) / yield_change["1990_1994"]
yield_change = yield_change.sort_values("change_pct", ascending=False)
display(yield_change.round(2))

**What this means:** this is a production trend, not a climate effect. Technology, crop varieties, inputs, policy and structural changes can improve yield at the same time as climate is changing.

### 2. What happens in warmer-than-trend years?

Raw temperature and raw yield both move over decades. I remove the linear time trend **inside each country × crop history** first.

For the chart only, I sample up to **12,000 points** so the figure stays readable. The sensitivity calculation later still uses all valid observations.

In [ ]:
plot_d = detrended.dropna(subset=["temp_detrended_c", "yield_detrended_t_ha"])
# Sampling affects only visual density, not the analysis.
plot_d = plot_d.sample(min(12000, len(plot_d)), random_state=42)

g = sns.lmplot(data=plot_d, x="temp_detrended_c", y="yield_detrended_t_ha", col="crop", col_wrap=3,
               height=3.3, aspect=1.2, scatter_kws={"alpha": 0.16, "s": 14},
               line_kws={"color": "green", "linewidth": 2})
g.set_axis_labels("Detrended temperature residual (°C)", "Detrended yield residual (t/ha)")
g.fig.suptitle("Interannual Temperature vs Yield After Removing Time Trends", y=1.03)
g.fig.savefig(FIG_DIR / "02_detrended_temperature_vs_yield.png", dpi=180, bbox_inches="tight")
plt.show()

### 3. Which crops look most temperature sensitive?

In [ ]:
valid_d = detrended.dropna(subset=["temp_detrended_c", "yield_detrended_t_ha"])
rows = []
for crop, part in valid_d.groupby("crop"):
    x = part["temp_detrended_c"].to_numpy(float)
    y = part["yield_detrended_t_ha"].to_numpy(float)
    if len(part) < 20 or np.std(x) < 1e-8:
        continue
    rows.append({"crop": crop, "observations": len(part), "yield_change_t_ha_per_1c": np.polyfit(x, y, deg=1)[0]})

sensitivity = pd.DataFrame(rows).sort_values("yield_change_t_ha_per_1c").reset_index(drop=True)
colors = ["#f7786b" if v < 0 else "#99ff99" for v in sensitivity["yield_change_t_ha_per_1c"]]

plt.figure(figsize=(10, 6))
ax = sns.barplot(data=sensitivity, x="crop", y="yield_change_t_ha_per_1c", hue="crop", palette=colors, legend=False)
plt.axhline(0, color="black", linewidth=1)
plt.title("Detrended Yield Association per +1°C")
plt.ylabel("Yield association (t/ha per +1°C)")
plt.xlabel("Crop")
for patch in ax.patches:
    h = patch.get_height()
    ax.annotate(f"{h:.3f}", (patch.get_x()+patch.get_width()/2, h), ha="center",
                va="bottom" if h >= 0 else "top", xytext=(0, 3 if h >= 0 else -3), textcoords="offset points", fontsize=9)
plt.tight_layout()
plt.savefig(FIG_DIR / "03_temperature_sensitivity.png", dpi=180, bbox_inches="tight")
plt.show()
display(sensitivity.round(4))

**Interpretation:** a negative slope means warmer-than-trend years tend to appear with yield below that production system's own trend. This is still an **association**, not a causal temperature effect.

### 4. Is there one perfect temperature?

I do **not** choose one exact observed degree and call it the optimum.

I use **8 broad temperature groups**. Eight gives enough detail to show broad patterns without pretending one exact value is scientifically proven to be best.

In [ ]:
temp_data = df.dropna(subset=["temperature_c", "yield_t_ha"]).copy()
# 8 quantile bins = broad exploratory ranges, not a physiological optimum.
temp_data["temp_bin"] = pd.qcut(temp_data["temperature_c"], q=8, duplicates="drop")
temp_summary = temp_data.groupby(["crop", "temp_bin"], observed=True)["yield_t_ha"].median().reset_index()

g = sns.catplot(data=temp_summary, x="temp_bin", y="yield_t_ha", col="crop", col_wrap=2,
                kind="bar", palette="Greens", sharex=False, sharey=False, height=3.5, aspect=1.4)
g.set_xticklabels(rotation=55, ha="right")
g.set_axis_labels("Observed temperature range", "Median yield (t/ha)")
g.fig.suptitle("Temperature Ranges — Exploratory, Not a Magic Degree", y=1.02)
g.fig.savefig(FIG_DIR / "04_temperature_ranges.png", dpi=180, bbox_inches="tight")
plt.show()

### 5. Does more rain always mean better yield?

In [ ]:
rain = df.dropna(subset=["precip_deviation_mm", "yield_yoy_pct_w"]).copy()
rain_plot = rain.sample(min(12000, len(rain)), random_state=42)

g = sns.lmplot(data=rain_plot, x="precip_deviation_mm", y="yield_yoy_pct_w", col="crop", col_wrap=3,
               height=3.3, aspect=1.2, scatter_kws={"alpha": 0.14, "s": 13}, line_kws={"color": "#66b3ff"})
g.set_axis_labels("Precipitation deviation from country mean (mm)", "YoY yield change (%)")
g.fig.suptitle("Rainfall Deviation vs Year-to-Year Yield Change", y=1.03)
g.fig.savefig(FIG_DIR / "05_rainfall_vs_yield_change.png", dpi=180, bbox_inches="tight")
plt.show()

### 6. What do irrigation and fertilizer tell us?

These are observational country-level indicators. I divide each available indicator into **four groups (quartiles)**: Low, Mid-low, Mid-high and High.

Quartiles make the comparison easy to read and avoid inventing arbitrary thresholds. Irrigation is shown only where it is actually observed.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, col, title in [(axes[0], "irrigated_land_pct", "Irrigation"), (axes[1], "fertilizer_kg_ha", "Fertilizer")]:
    x = df.dropna(subset=[col, "yield_t_ha"]).copy()
    x["group"] = pd.qcut(x[col], 4, labels=["Low", "Mid-low", "Mid-high", "High"], duplicates="drop")
    sns.boxplot(data=x, x="group", y="yield_t_ha", hue="group", palette="Set2", legend=False, showfliers=False, ax=ax)
    ax.set_title(f"Yield by {title} Intensity — Descriptive")
    ax.set_xlabel(f"{title} group")
    ax.set_ylabel("Yield (t/ha)")
plt.tight_layout()
plt.savefig(FIG_DIR / "06_management_groups.png", dpi=180, bbox_inches="tight")
plt.show()
print(f"Irrigation coverage: {100*df['irrigated_land_pct'].notna().mean():.2f}%")
print(f"Fertilizer coverage: {100*df['fertilizer_kg_ha'].notna().mean():.2f}%")

### 7. Where is climate risk highest?

The V1 risk screen combines:

1. **Detrended yield volatility** — instability around a country's crop trend.
2. **Warming penalty** — the negative part of its detrended temperature-yield slope.

I require at least **15 observations** before a country-crop history enters the ranking. This avoids ranking very short histories too aggressively.

The score is for **prioritization**, not an insurance probability.

In [ ]:
MIN_RISK_OBSERVATIONS = 15
risk_base = detrended.groupby(["Code", "Entity", "crop"]).agg(
    yield_mean=("yield_t_ha", "mean"), detrended_yield_std=("yield_detrended_t_ha", "std"), observations=("yield_t_ha", "size")
).reset_index()

risk_slopes=[]
for (code_value, crop), part in valid_d.groupby(["Code", "crop"]):
    if len(part) < MIN_RISK_OBSERVATIONS or part["temp_detrended_c"].std() < 1e-8:
        continue
    slope = np.polyfit(part["temp_detrended_c"].to_numpy(float), part["yield_detrended_t_ha"].to_numpy(float), deg=1)[0]
    risk_slopes.append({"Code": code_value, "crop": crop, "temp_slope": slope})

risk = risk_base.merge(pd.DataFrame(risk_slopes), on=["Code", "crop"], how="inner")
risk = risk[(risk["observations"] >= MIN_RISK_OBSERVATIONS) & (risk["yield_mean"] > 0) & risk["detrended_yield_std"].notna()].copy()
risk["volatility_cv"] = risk["detrended_yield_std"] / risk["yield_mean"]
risk["warming_penalty"] = (-risk["temp_slope"]).clip(lower=0)
risk["volatility_rank"] = risk["volatility_cv"].rank(pct=True)
risk["warming_penalty_rank"] = risk["warming_penalty"].rank(pct=True)
risk["risk_score"] = (risk["volatility_rank"] + risk["warming_penalty_rank"]) / 2
risk = risk.sort_values(["risk_score", "warming_penalty"], ascending=[False, False]).reset_index(drop=True)

top_risk = risk.head(10).copy()
plt.figure(figsize=(11, 7))
sns.scatterplot(data=risk, x="warming_penalty", y="volatility_cv", hue="crop", size="risk_score", sizes=(25, 180), palette="Set2", alpha=0.65)
for _, row in top_risk.head(8).iterrows():
    plt.annotate(f"{row['Entity']} – {row['crop']}", (row["warming_penalty"], row["volatility_cv"]), xytext=(5,5), textcoords="offset points", fontsize=8)
plt.title("Climate Risk Screen — Where Should We Investigate First?")
plt.xlabel("Negative detrended temperature association")
plt.ylabel("Detrended yield volatility")
plt.tight_layout()
plt.savefig(FIG_DIR / "07_risk_screen.png", dpi=180, bbox_inches="tight")
plt.show()
display(top_risk[["Entity", "crop", "risk_score", "volatility_cv", "temp_slope", "observations"]].round(4))

### 8. Can climate information beat strong forecasting baselines?

I use a **time split**, not a random split:

- **Train:** 1990–2017
- **Test:** 2018–2023

Why 2018? It leaves the most recent six years as a realistic future holdout while keeping a long training history.

### Model parameters

- **250 trees:** stable enough without making this small project unnecessarily heavy.
- **min_samples_leaf = 5:** smooths noisy leaf rules.
- **random_state = 42:** reproducibility.

I did **not** tune these parameters against the 2018+ test set.

I compare with three baselines: crop median, country × crop median, and persistence (last known pre-2018 yield).

In [ ]:
SPLIT_YEAR = 2018
RANDOM_STATE = 42
SOURCE_FEATURES = ["temperature_c", "precipitation_mm", "fertilizer_kg_ha"]
NUMERIC_FEATURES = ["temp_anomaly_c", "precip_anomaly_mm", "fertilizer_anomaly_kg_ha"]

model_df = df.dropna(subset=["yield_t_ha"]).copy()
train = model_df[model_df["Year"] < SPLIT_YEAR].copy()
test = model_df[model_df["Year"] >= SPLIT_YEAR].copy()

global_median = train["yield_t_ha"].median()
crop_medians = train.groupby("crop")["yield_t_ha"].median()
crop_baseline = test["crop"].map(crop_medians).fillna(global_median).to_numpy()
cc_medians = train.groupby(["Code", "crop"])["yield_t_ha"].median()

def map_country_crop(frame, values):
    keys = pd.MultiIndex.from_frame(frame[["Code", "crop"]])
    return values.reindex(keys).to_numpy(dtype=float)

country_crop_baseline = map_country_crop(test, cc_medians)
country_crop_baseline = np.where(np.isnan(country_crop_baseline), crop_baseline, country_crop_baseline)
train_crop_baseline = train["crop"].map(crop_medians).fillna(global_median).to_numpy()
train_cc_reference = map_country_crop(train, cc_medians)
train_cc_reference = np.where(np.isnan(train_cc_reference), train_crop_baseline, train_cc_reference)

last_pre2018 = train.sort_values(["Code", "crop", "Year"]).groupby(["Code", "crop"], as_index=False).tail(1).set_index(["Code", "crop"])["yield_t_ha"]
persistence = map_country_crop(test, last_pre2018)
persistence = np.where(np.isnan(persistence), country_crop_baseline, persistence)

# Train-only normals avoid future-data leakage.
unique_train = train[["Code", "Year"] + SOURCE_FEATURES].drop_duplicates(["Code", "Year"])
train_normals = unique_train.groupby("Code")[SOURCE_FEATURES].mean()
global_normals = unique_train[SOURCE_FEATURES].mean()

for source, target in {"temperature_c":"temp_anomaly_c", "precipitation_mm":"precip_anomaly_mm", "fertilizer_kg_ha":"fertilizer_anomaly_kg_ha"}.items():
    train[target] = train[source] - train["Code"].map(train_normals[source]).fillna(global_normals[source])
    test[target] = test[source] - test["Code"].map(train_normals[source]).fillna(global_normals[source])

preprocessor = ColumnTransformer([
    ("num", Pipeline([("imputer", SimpleImputer(strategy="median", add_indicator=True))]), NUMERIC_FEATURES),
    ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=True))]), ["crop"]),
])

model = Pipeline([
    ("prep", preprocessor),
    ("model", RandomForestRegressor(n_estimators=250, min_samples_leaf=5, random_state=RANDOM_STATE, n_jobs=-1)),
])

X_train = train[NUMERIC_FEATURES + ["crop"]]
X_test = test[NUMERIC_FEATURES + ["crop"]]
y_train_residual = train["yield_t_ha"].to_numpy() - train_cc_reference
model.fit(X_train, y_train_residual)
residual_prediction = model.predict(X_test)
prediction = country_crop_baseline + residual_prediction
y_test = test["yield_t_ha"].to_numpy()

metrics = {
    "Crop median": mean_absolute_error(y_test, crop_baseline),
    "Country × crop median": mean_absolute_error(y_test, country_crop_baseline),
    "Climate-anomaly model": mean_absolute_error(y_test, prediction),
    "Persistence": mean_absolute_error(y_test, persistence),
}
model_r2 = r2_score(y_test, prediction)
metrics_df = pd.Series(metrics, name="MAE_t_ha").to_frame().sort_values("MAE_t_ha")
display(metrics_df.round(4))
print(f"Model R²: {model_r2:.4f}")

### Model vs baselines

Lower MAE is better. This chart shows whether the more complicated model actually adds value.

In [ ]:
plot_metrics = metrics_df.reset_index(); plot_metrics.columns = ["Method", "MAE_t_ha"]
plt.figure(figsize=(9, 5))
ax = sns.barplot(data=plot_metrics, x="Method", y="MAE_t_ha", hue="Method", palette="Set2", legend=False)
plt.title("2018+ Holdout — Model vs Strong Baselines")
plt.ylabel("MAE (t/ha)"); plt.xlabel("")
for patch in ax.patches:
    h = patch.get_height(); ax.annotate(f"{h:.3f}", (patch.get_x()+patch.get_width()/2, h), ha="center", va="bottom", xytext=(0,3), textcoords="offset points")
plt.xticks(rotation=12); plt.tight_layout()
plt.savefig(FIG_DIR / "08_model_vs_baselines.png", dpi=180, bbox_inches="tight")
plt.show()

### 9. Where does the model fail?

In [ ]:
pred = test[["Entity", "Code", "Year", "crop", "yield_t_ha"]].copy()
pred["predicted_yield_t_ha"] = prediction
pred["persistence_yield_t_ha"] = persistence
pred["abs_error"] = np.abs(pred["yield_t_ha"] - pred["predicted_yield_t_ha"])

p = pred.sample(min(4000, len(pred)), random_state=42)
plt.figure(figsize=(9, 7))
sns.scatterplot(data=p, x="yield_t_ha", y="predicted_yield_t_ha", hue="crop", palette="Set2", alpha=0.6)
lim = max(pred["yield_t_ha"].max(), pred["predicted_yield_t_ha"].max())
plt.plot([0,lim], [0,lim], "--", color="black", linewidth=1)
plt.title("Actual vs Predicted Yield — 2018+ Holdout")
plt.xlabel("Actual yield (t/ha)"); plt.ylabel("Predicted yield (t/ha)")
plt.tight_layout(); plt.savefig(FIG_DIR / "09_actual_vs_predicted.png", dpi=180, bbox_inches="tight"); plt.show()

error_by_crop = pred.groupby("crop")["abs_error"].agg(["mean", "median", "count"]).sort_values("mean", ascending=False)
display(error_by_crop.round(4))

In [ ]:
error_plot = error_by_crop["mean"].rename("MAE_t_ha").reset_index()
plt.figure(figsize=(9, 5))
ax = sns.barplot(data=error_plot, x="crop", y="MAE_t_ha", hue="crop", palette="Set2", legend=False)
plt.title("Model Error by Crop")
plt.ylabel("Mean absolute error (t/ha)"); plt.xlabel("Crop")
for patch in ax.patches:
    h = patch.get_height(); ax.annotate(f"{h:.2f}", (patch.get_x()+patch.get_width()/2, h), ha="center", va="bottom", xytext=(0,3), textcoords="offset points")
plt.tight_layout(); plt.savefig(FIG_DIR / "10_error_by_crop.png", dpi=180, bbox_inches="tight"); plt.show()

## 6. Final Business Takeaways

In [ ]:
best_crop = yield_change["change_pct"].idxmax(); best_crop_change = yield_change.loc[best_crop, "change_pct"]
strongest_temp = sensitivity.iloc[0]
model_mae = metrics["Climate-anomaly model"]; persistence_mae = metrics["Persistence"]; cc_mae = metrics["Country × crop median"]
improvement_vs_cc = 100 * (cc_mae - model_mae) / cc_mae

summary = f"""
### What I found

- The final panel contains **{len(df):,} country-year-crop rows** across **{df['Code'].nunique()} countries / territories** and **{df['crop'].nunique()} crops**.
- **{best_crop}** has the largest median yield increase: **{best_crop_change:.1f}%** comparing 1990–1994 with 2019–2023.
- After detrending, the strongest pooled negative temperature association is **{strongest_temp['crop']} ({strongest_temp['yield_change_t_ha_per_1c']:.4f} t/ha per +1°C)**.
- The top risk-screen segment is **{risk.iloc[0]['Entity']} – {risk.iloc[0]['crop']}**.
- The climate-anomaly model reaches **MAE {model_mae:.4f} t/ha** and **R² {model_r2:.4f}** on 2018+.
- It improves on the static country × crop median by **{improvement_vs_cc:.2f}%**.
- But persistence is stronger at **{persistence_mae:.4f} MAE**.

### Main client message

**Recent production history is still a stronger short-term predictor than annual climate anomalies alone.**

That is useful information. The model does not need to win for the analysis to create value.
"""
display(Markdown(summary))

## 7. Limitations

This project works at **country-year level**, so it cannot see everything happening on a real farm.

Important limitations:

- annual climate does not show individual heat waves,
- total rainfall does not show when the rain arrived,
- country averages hide local soil and farm conditions,
- irrigation coverage is limited,
- fertilizer and irrigation are observational variables,
- detrending reduces time-trend confounding but does not establish causality.

The clearest next upgrade is to add **growing-season-specific climate, crop calendars and more local agricultural data**.

## Conclusion

The project gives a simple decision story:

**Where did yield change? → Where does climate pressure appear? → Which segments deserve attention? → Does climate data improve prediction?**

The strongest result is not that a complicated model wins.

The strongest result is that the project tests the idea honestly and shows where the data adds value — and where a simple historical baseline is still better.

**Final client message:** climate risk cannot be removed, but it can be measured more carefully and prioritized better.

## Saved Figures

All notebook figures are also saved during the run in:

`/content/climate_crop_figures/`

This makes it easy to reuse the strongest visuals later in the GitHub README or presentation.